In [28]:
import numpy as np
from LanzaModels import TVL2_1D
from ADMMsRustici import SRASpectralSolver
from signalClass import *
import time

In [29]:
np.random.seed(24102000)
n = 512

construct blur matrix

In [30]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [31]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 50)
RndSignal.generate_GG_realization(0, sigma, 1)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL2 model

In [32]:
mu = 2
VarModel = TVL2_1D.TVL2_1DClass(A, xCorrupted, mu)

Now, we need to initialize and define the solver

In [33]:
#begin solver construction
np.random.seed(24102002)

xk = np.random.randn(n,)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

x0 = xk.copy()
y0 = yk.copy()

MySolver = SRASpectralSolver.SRASolverClass(VarModel, xk, yk, lk, betak)

#end solver construction

In [34]:
iters = 700

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))
ImgHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [35]:
timer = 0

for iter in range(0, iters):

    print(f"{iter + 1} / {iters}")

    sTime = time.perf_counter_ns()

    xk_1, yk_1, lk_1, betak_1 = MySolver.CallIterationStep(xk, yk, lk, betak)

    eTime = time.perf_counter_ns()

    timer += ( (eTime - sTime) / 1e9 )

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    dualResidue = betak_1 * np.linalg.norm(VarModel.P.T @ (VarModel.Q @ (yk - yk_1)) )

    XsolutionHistory[iter, :] = xk_1
    YsolutionHistory[iter, :] = yk_1
    lambdaHistory[iter, :] = lk_1
    betaHistory[iter] = betak_1

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResidue
    ImgHistory[iter] = VarModel(xk_1)
    CpuTimes[iter] = timer

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1

    if (max(primalResidue, dualResidue) <= 1e-9):
        print(iter)
        break


1 / 700
2 / 700
3 / 700
4 / 700
5 / 700
6 / 700
7 / 700
8 / 700
9 / 700
10 / 700
11 / 700
12 / 700
13 / 700
14 / 700
15 / 700
16 / 700
17 / 700
18 / 700
19 / 700
20 / 700
21 / 700
22 / 700
23 / 700
24 / 700
25 / 700
26 / 700
27 / 700
28 / 700
29 / 700
30 / 700
31 / 700
32 / 700
33 / 700
34 / 700
35 / 700
36 / 700
37 / 700
38 / 700
39 / 700
40 / 700
41 / 700
42 / 700
43 / 700
44 / 700
45 / 700
46 / 700
47 / 700
48 / 700
49 / 700
50 / 700
51 / 700
52 / 700
53 / 700
54 / 700
55 / 700
56 / 700
57 / 700
58 / 700
59 / 700
60 / 700
61 / 700
62 / 700
63 / 700
64 / 700
65 / 700
66 / 700
67 / 700
68 / 700
69 / 700
70 / 700
71 / 700
72 / 700
73 / 700
74 / 700
75 / 700
76 / 700
77 / 700
78 / 700
79 / 700
80 / 700
81 / 700
82 / 700
83 / 700
84 / 700
85 / 700
86 / 700
87 / 700
88 / 700
89 / 700
90 / 700
91 / 700
92 / 700
93 / 700
94 / 700
95 / 700
96 / 700
97 / 700
98 / 700
99 / 700
100 / 700
101 / 700
102 / 700
103 / 700
104 / 700
105 / 700
106 / 700
107 / 700
108 / 700
109 / 700
110 / 700
111 / 70

In [36]:
np.savez_compressed(
    "./SRAADMMTVL2-Laplace.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,
	IMGs = ImgHistory,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
	x0 = x0,
	y0 = y0
)